In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import torch

from transformers_arch.attention_mechanism import scaled_dot_product_attention

torch.manual_seed(42)
longueur = 4
q = torch.randn(1, longueur, 8)
k = torch.randn(1, longueur, 8)
v = torch.randn(1, longueur, 8)

# Masque causal : le mot i ne doit voir QUE les mots 0..i (pas le futur)
masque_causal = torch.triu(torch.ones(longueur, longueur), diagonal=1).bool()
print("Masque causal (True = position interdite) :")
print(masque_causal.numpy())

_, poids = scaled_dot_product_attention(q, k, v, masque_causal.unsqueeze(0))
print("\nMatrice de poids resultante (doit etre triangulaire) :")
print(poids[0].detach().numpy().round(3))
print("\n-> le mot 0 ne voit QUE lui-meme, le mot 3 voit tout le monde")

Masque causal (True = position interdite) :
[[False  True  True  True]
 [False False  True  True]
 [False False False  True]
 [False False False False]]

Matrice de poids resultante (doit etre triangulaire) :
[[1.    0.    0.    0.   ]
 [0.059 0.941 0.    0.   ]
 [0.211 0.418 0.371 0.   ]
 [0.193 0.18  0.195 0.432]]

-> le mot 0 ne voit QUE lui-meme, le mot 3 voit tout le monde


## 1.   Prompting

In [2]:
# --- BLOC 1 : construction des 3 types de prompts ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from llms.prompting import (
    build_chain_of_thought_prompt,
    build_few_shot_prompt,
    build_zero_shot_prompt,
    classify_with_llm,
)

avis = "The delivery was slow but the product quality is excellent"

print("=== ZERO-SHOT ===")
prompt_zero_shot = build_zero_shot_prompt(avis)
print(prompt_zero_shot)

=== ZERO-SHOT ===
Classify the sentiment of this review as positive, negative, or mixed:
"The delivery was slow but the product quality is excellent"
Sentiment:


In [3]:
# --- BLOC 2 : few-shot avec des exemples ABSA-pertinents ---
exemples = [
    ("Amazing quality, fast shipping!", "positive"),
    ("Terrible service, arrived broken", "negative"),
    ("Great product but the box was damaged", "mixed"),
]
prompt_few_shot = build_few_shot_prompt(avis, exemples)
print("\n=== FEW-SHOT ===")
print(prompt_few_shot)


=== FEW-SHOT ===
Review: "Amazing quality, fast shipping!" -> Sentiment: positive
Review: "Terrible service, arrived broken" -> Sentiment: negative
Review: "Great product but the box was damaged" -> Sentiment: mixed
Review: "The delivery was slow but the product quality is excellent" -> Sentiment:


In [4]:
# --- BLOC 3 : chain-of-thought pour un avis multi-aspects ---
prompt_cot = build_chain_of_thought_prompt(avis)
print("\n=== CHAIN-OF-THOUGHT ===")
print(prompt_cot)


=== CHAIN-OF-THOUGHT ===
Review: "The delivery was slow but the product quality is excellent"
Let's think step by step:
1. Identify the aspects mentioned in this review.
2. Determine the sentiment for each aspect separately.
3. Combine into an overall sentiment (positive, negative, or mixed).
Answer:


In [5]:
# --- BLOC 4 : appel reel a un LLM generatif (necessite internet) ---
print("\n=== APPEL REEL AU MODELE (FLAN-T5) ===")
reponse_zero_shot = classify_with_llm(prompt_zero_shot)
print("Zero-shot  ->", reponse_zero_shot)

reponse_few_shot = classify_with_llm(prompt_few_shot)
print("Few-shot   ->", reponse_few_shot)

reponse_cot = classify_with_llm(prompt_cot)
print("Chain-of-thought ->", reponse_cot)

# A verifier chez toi : le few-shot et le chain-of-thought donnent-ils
# une reponse plus precise ("mixed", capturant les 2 aspects) que le
# zero-shot seul (qui pourrait se contenter d'un "positive" ou
# "negative" global, sans nuance) ?


=== APPEL REEL AU MODELE (FLAN-T5) ===


/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2437.03it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Zero-shot  -> positive


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2533.52it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Few-shot   -> positive


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2172.63it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Chain-of-thought -> positive


In [6]:
# --- BLOC 5 : tester sur plusieurs avis, comparer les 3 approches ---
avis_test = [
    "Absolutely perfect in every way",
    "Complete waste of money, terrible",
    "Good product but customer service was rude",
]

print("\n=== COMPARAISON SUR 3 AVIS ===")
for a in avis_test:
    p_zs = build_zero_shot_prompt(a)
    p_cot = build_chain_of_thought_prompt(a)
    r_zs = classify_with_llm(p_zs)
    r_cot = classify_with_llm(p_cot)
    print(f"\nAvis : {a}")
    print(f"  Zero-shot        : {r_zs}")
    print(f"  Chain-of-thought : {r_cot}")


=== COMPARAISON SUR 3 AVIS ===


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2711.21it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1450.01it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Avis : Absolutely perfect in every way
  Zero-shot        : positive
  Chain-of-thought : positive


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2522.10it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2718.75it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Avis : Complete waste of money, terrible
  Zero-shot        : negative
  Chain-of-thought : negative


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2630.20it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2529.68it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Avis : Good product but customer service was rude
  Zero-shot        : positive
  Chain-of-thought : positive


## 2.    RAG    (Retrieval Augmented generation)

In [2]:
# --- BLOC 1 : recherche par similarite, sans reseau ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import numpy as np

from llms.rag import build_rag_prompt, find_most_similar

documents = [
    "La livraison est rapide",
    "Le produit est excellent",
    "Le service client est lent",
    "Emballage tres soigne",
]
embeddings_docs = np.array(
    [
        [0.9, 0.1, 0.0],
        [0.1, 0.9, 0.0],
        [0.85, 0.15, 0.1],
        [0.0, 0.1, 0.9],
    ]
)
embedding_question = np.array([0.88, 0.12, 0.05])

resultats = find_most_similar(embedding_question, embeddings_docs, top_k=2)
print("Documents les plus pertinents (vecteurs synthetiques) :")
for i, score in resultats:
    print(f"  {score:.3f}  {documents[i]}")

Documents les plus pertinents (vecteurs synthetiques) :
  0.998  La livraison est rapide
  0.997  Le service client est lent


In [3]:
# --- BLOC 2 : vrai retrieval avec sentence-transformers ---
from llms.rag import build_document_index, retrieve_relevant_documents

avis_clients = [
    "The delivery was super fast, arrived next day",
    "Product quality is outstanding, highly recommend",
    "Customer service took forever to respond",
    "Packaging was damaged but the product inside was fine",
    "Refund process was quick and painless",
    "The website checkout was confusing and slow",
]

embeddings, modele_embedding = build_document_index(avis_clients)
print("\nForme des embeddings des avis :", embeddings.shape)

question = "What do customers say about delivery speed?"
resultats_reels = retrieve_relevant_documents(
    question, avis_clients, embeddings, modele_embedding, top_k=2
)
print(f"\nQuestion : {question}")
print("Avis retrouves :")
for avis, score in resultats_reels:
    print(f"  {score:.3f}  {avis}")

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3101.67it/s]



Forme des embeddings des avis : (6, 384)

Question : What do customers say about delivery speed?
Avis retrouves :
  0.633  The delivery was super fast, arrived next day
  0.469  Customer service took forever to respond


In [ ]:
# --- BLOC 3 : construire le prompt final ---
prompt_rag = build_rag_prompt(question, [a for a, _ in resultats_reels])
print("\nPrompt RAG complet :")
print(prompt_rag)


Prompt RAG complet :
Context (customer reviews):
- The delivery was super fast, arrived next day
- Customer service took forever to respond

Question: What do customers say about delivery speed?
Answer based only on the context above:


In [5]:
# --- BLOC 4 : pipeline RAG complet, retrieval + generation ---
from llms.prompting import classify_with_llm
from llms.rag import answer_with_rag


def generer_reponse(prompt):
    return classify_with_llm(prompt, model_name="google/flan-t5-base")


resultat_final = answer_with_rag(
    question,
    avis_clients,
    embeddings,
    modele_embedding,
    generer_reponse,
    top_k=2,
)
print("\n=== REPONSE FINALE DU RAG ===")
print(resultat_final["answer"])

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2439.26it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



=== REPONSE FINALE DU RAG ===
The delivery was super fast, arrived next day


In [6]:
# --- BLOC 5 : comparer avec une question SANS rapport aux documents ---
question_hors_sujet = "What is the capital of France?"
resultats_hors_sujet = retrieve_relevant_documents(
    question_hors_sujet, avis_clients, embeddings, modele_embedding, top_k=2
)
print(f"\nQuestion hors-sujet : {question_hors_sujet}")
print("Documents quand meme retrouves (scores probablement bas) :")
for avis, score in resultats_hors_sujet:
    print(f"  {score:.3f}  {avis}")


Question hors-sujet : What is the capital of France?
Documents quand meme retrouves (scores probablement bas) :
  0.071  The website checkout was confusing and slow
  0.036  The delivery was super fast, arrived next day


In [16]:
# --- BLOC 6 : ingestion d'un vrai PDF -- extraction et chunking ---
# necessite : uv add pypdf
from llms.rag import chunk_text, extract_text_from_pdf, load_pdf_as_documents

# remplace par le chemin de ton propre PDF (rapport, export d'avis...)
chemin_pdf = "../data/raw/avis_clients.pdf"
texte_pdf = extract_text_from_pdf(chemin_pdf)
print("Texte extrait (200 premiers caracteres) :")
print(texte_pdf[:200])

chunks_pdf = chunk_text(texte_pdf, chunk_size=200, overlap=50)
print(f"\n{len(chunks_pdf)} chunks obtenus")
print("Premier chunk :", chunks_pdf[1][:150], "...")

Texte extrait (200 premiers caracteres) :
GlobaTrend Insights
Customer Reviews Export (English)
Review #1
The delivery was super fast, arrived the next day. Really impressed with how quickly everything was
processed and shipped from the wareh

6 chunks obtenus
Premier chunk : checkout process was confusing and slow. I almost gave up trying to complete my order due to the clunky interface and repeated error messages. Review  ...


In [13]:
from llms.prompting import classify_with_llm
from llms.rag import (
    answer_with_rag,
    build_document_index,
    retrieve_relevant_documents,
)

documents_pdf = load_pdf_as_documents(chemin_pdf, chunk_size=30, overlap=10)
embeddings_pdf, modele_pdf = build_document_index(documents_pdf)

question_pdf = "What do customers say about delivery?"


def generer_reponse(prompt):
    return classify_with_llm(prompt, model_name="google/flan-t5-large")


resultat_pdf = answer_with_rag(
    question_pdf,
    documents_pdf,
    embeddings_pdf,
    modele_pdf,
    generer_reponse,
    top_k=3,
)

print("\n=== ETAPE 1 : chunks retrouves (juste du texte selectionne) ===")
for chunk, score in resultat_pdf["retrieved_documents"]:
    print(f"  {score:.3f}  {chunk[:100]}...")

print("\n=== ETAPE 2 : prompt envoye au LLM (question + chunks) ===")
print(resultat_pdf["prompt"][:300], "...")

print("\n=== ETAPE 3 : REPONSE GENEREE PAR LE LLM (la vraie reponse) ===")
print(resultat_pdf["answer"])

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 2716.94it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



=== ETAPE 1 : chunks retrouves (juste du texte selectionne) ===
  0.508  Packaging was excellent, everything was wrapped carefully and arrived in perfect condition. You can ...
  0.488  GlobaTrend Insights Customer Reviews Export (English) Review #1 The delivery was super fast, arrived...
  0.479  the price point. Review #3 Customer service took forever to respond to my email. I waited almost a w...

=== ETAPE 2 : prompt envoye au LLM (question + chunks) ===
Context (customer reviews):
- Packaging was excellent, everything was wrapped carefully and arrived in perfect condition. You can tell real care went into how this was shipped. Review #13 Prices have gone up noticeably
- GlobaTrend Insights Customer Reviews Export (English) Review #1 The delivery wa ...

=== ETAPE 3 : REPONSE GENEREE PAR LE LLM (la vraie reponse) ===
The delivery was super fast, arrived the next day. Really impressed with how quickly everything was processed and shipped from the warehouse - the price point
